# 🚀 TabDrift v2: Corrected Normalization + EMA

Key changes in this version:
- **Fix 1**: Latent normalization now uses `(z - mean) / 2` to match TabSyn's geometry
- **Fix 2**: EMA model averaging (decay=0.999) for stable generation
- **Fix 3**: EMA weights automatically loaded at inference time

In [ ]:
# 1. Clone Private GitHub Repository
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
github_token = user_secrets.get_secret("GITHUB_TOKEN")

GITHUB_USER = "ahmed-fouad-lagha"
REPO_NAME = "tabsyn"

!git clone https://{github_token}@github.com/{GITHUB_USER}/{REPO_NAME}.git
%cd {REPO_NAME}

In [ ]:
# 2. Install Dependencies
!pip install -q icecream zero tomli tomli-w category_encoders executing asttokens prdc catboost lightgbm

In [ ]:
# 3. Verify GPU
!nvidia-smi

In [ ]:
# 4. Train TabDrift v2 (corrected normalization + EMA)
!PYTHONPATH=. python tabsyn/drift_train.py \
    --dataname adult \
    --gpu 0 \
    --epochs 4000 \
    --batch_size 4096 \
    --lr 1e-4 \
    --hidden_size 1024 \
    --temperatures 0.1 0.5 1.0 2.0 \
    --drift_scale 1.5 \
    --patience 4000

In [ ]:
# 5. Generate synthetic data (auto-loads EMA weights)
!PYTHONPATH=. python tabsyn/drift_sample.py \
    --dataname adult \
    --gpu 0 \
    --steps 1

In [ ]:
# 6. Evaluate
!python eval/eval_mle.py --dataname adult --model tabdrift

In [ ]:
# 7. Print results
import json
with open('eval/mle/adult/tabdrift.json', 'r') as f:
    scores = json.load(f)
print(json.dumps(scores, indent=4))